# 🚀 LoveDA Adaptive Super-Resolution Pipeline — Continuation Notebook (ResNet-50)
This notebook seamlessly resumes from saved checkpoints from `Updated50` or `notebooka695a1984d`. If training reached epoch 80, it automatically skips retraining and performs the full downstream task evaluation, ablation comparison, metrics (PSNR, SSIM, mIoU), and final comparison grid generation.

In [ ]:
import os, sys, time, glob, random, cv2
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from skimage.metrics import structural_similarity as ssim_fn

!pip install -q opencv-python-headless fvcore einops scikit-image

print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
LOVEDA_ROOT = '/kaggle/input/loveda-raw/loveda_zips'
if not os.path.exists(LOVEDA_ROOT):
    found = glob.glob('/kaggle/input/**/loveda_zips', recursive=True)
    if found:
        LOVEDA_ROOT = found[0]

def build_file_lists(root, split_folder='Train'):
    img_paths, mask_paths = [], []
    for scene in ['Urban', 'Rural']:
        img_dir = os.path.join(root, split_folder, split_folder, scene, 'images_png')
        mask_dir = os.path.join(root, split_folder, split_folder, scene, 'masks_png')
        if os.path.isdir(img_dir):
            for p in sorted(glob.glob(os.path.join(img_dir, '*.png'))):
                fname = os.path.basename(p)
                mp = os.path.join(mask_dir, fname)
                if os.path.exists(mp):
                    img_paths.append(p)
                    mask_paths.append(mp)
    return img_paths, mask_paths

train_imgs, train_masks = build_file_lists(LOVEDA_ROOT, 'Train')
val_imgs, val_masks = build_file_lists(LOVEDA_ROOT, 'Val')
print(f"Train pairs: {len(train_imgs)} | Val pairs: {len(val_masks)}")

IGNORE_INDEX = 0
HIGH_IMPORTANCE_CLASSES = {2, 3}  # building, road
NUM_CLASSES = 8
HR_PATCH = 256
SCALE = 4
LR_PATCH = HR_PATCH // SCALE  # 64

def degrade(hr_patch, scale=SCALE, blur_sigma=1.5, noise_std=2.0):
    blurred = cv2.GaussianBlur(hr_patch, (0, 0), sigmaX=blur_sigma)
    h, w = hr_patch.shape[:2]
    lr = cv2.resize(blurred, (w // scale, h // scale), interpolation=cv2.INTER_CUBIC)
    noise = np.random.normal(0, noise_std, lr.shape)
    return np.clip(lr.astype(np.float32) + noise, 0, 255).astype(np.uint8)

class LoveDASRDataset(Dataset):
    def __init__(self, image_paths, mask_paths, patch_size=HR_PATCH, augment=True):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.patch_size = patch_size
        self.augment = augment

    def __len__(self):
        return len(self.image_paths)

    def _random_crop(self, img, mask):
        h, w = img.shape[:2]
        ps = self.patch_size
        if h < ps or w < ps:
            pad_h, pad_w = max(0, ps - h), max(0, ps - w)
            img = cv2.copyMakeBorder(img, 0, pad_h, 0, pad_w, cv2.BORDER_REFLECT)
            mask = cv2.copyMakeBorder(mask, 0, pad_h, 0, pad_w, cv2.BORDER_REFLECT)
            h, w = img.shape[:2]
        top = random.randint(0, h - ps)
        left = random.randint(0, w - ps)
        return img[top:top+ps, left:left+ps], mask[top:top+ps, left:left+ps]

    def _augment(self, img, mask):
        if random.random() < 0.5: img, mask = np.fliplr(img).copy(), np.fliplr(mask).copy()
        if random.random() < 0.5: img, mask = np.flipud(img).copy(), np.flipud(mask).copy()
        k = random.choice([0, 1, 2, 3])
        if k: img, mask = np.rot90(img, k).copy(), np.rot90(mask, k).copy()
        return img, mask

    def __getitem__(self, idx):
        img = cv2.cvtColor(cv2.imread(self.image_paths[idx]), cv2.COLOR_BGR2RGB)
        mask = cv2.imread(self.mask_paths[idx], cv2.IMREAD_GRAYSCALE)
        hr, mask = self._random_crop(img, mask)
        if self.augment: hr, mask = self._augment(hr, mask)
        lr = degrade(hr)
        importance = np.isin(mask, list(HIGH_IMPORTANCE_CLASSES)).astype(np.float32)
        return {
            'lr': torch.from_numpy(lr).permute(2, 0, 1).float() / 255.0,
            'hr': torch.from_numpy(hr).permute(2, 0, 1).float() / 255.0,
            'mask': torch.from_numpy(mask.astype(np.int64)),
            'importance': torch.from_numpy(importance).unsqueeze(0)
        }

val_dataset = LoveDASRDataset(val_imgs, val_masks, patch_size=HR_PATCH, augment=False)
train_dataset = LoveDASRDataset(train_imgs, train_masks, patch_size=HR_PATCH, augment=True)

In [ ]:
import torchvision.models as models

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.block(x)

class RegionImportanceNet(nn.Module):
    def __init__(self, base_ch=32, scale=SCALE):
        super().__init__()
        self.scale = scale
        self.enc1 = ConvBlock(3, base_ch)
        self.enc2 = ConvBlock(base_ch, base_ch*2)
        self.enc3 = ConvBlock(base_ch*2, base_ch*4)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = ConvBlock(base_ch*4, base_ch*8)
        self.up3 = nn.ConvTranspose2d(base_ch*8, base_ch*4, 2, stride=2)
        self.dec3 = ConvBlock(base_ch*8, base_ch*4)
        self.up2 = nn.ConvTranspose2d(base_ch*4, base_ch*2, 2, stride=2)
        self.dec2 = ConvBlock(base_ch*4, base_ch*2)
        self.up1 = nn.ConvTranspose2d(base_ch*2, base_ch, 2, stride=2)
        self.dec1 = ConvBlock(base_ch*2, base_ch)
        self.out_conv = nn.Conv2d(base_ch, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b  = self.bottleneck(self.pool(e3))
        d3 = self.dec3(torch.cat([self.up3(b), e3], 1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], 1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], 1))
        importance_lr = torch.sigmoid(self.out_conv(d1))
        return F.interpolate(importance_lr, scale_factor=self.scale, mode='bilinear', align_corners=False)

class ResNetSegmenter(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, backbone_name='resnet50'):
        super().__init__()
        if backbone_name == 'resnet50':
            backbone = models.resnet50(weights=None)
        elif backbone_name == 'resnet152':
            backbone = models.resnet152(weights=None)
        else:
            raise ValueError(f"Unknown backbone: {backbone_name}")
        self.enc0 = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu)
        self.maxpool = backbone.maxpool
        self.enc1 = backbone.layer1
        self.enc2 = backbone.layer2
        self.enc3 = backbone.layer3
        self.enc4 = backbone.layer4
        in_ch = 2048 if backbone_name in ['resnet50', 'resnet152'] else 512
        self.up4 = nn.ConvTranspose2d(in_ch, 256, 2, stride=2)
        self.dec4 = ConvBlock(256 + 1024, 256)
        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec3 = ConvBlock(128 + 512, 128)
        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec2 = ConvBlock(64 + 256, 64)
        self.up1 = nn.ConvTranspose2d(64, 64, 2, stride=2)
        self.dec1 = ConvBlock(64 + 64, 64)
        self.final_up = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.final_block = ConvBlock(32, 32)
        self.out_conv = nn.Conv2d(32, num_classes, 1)

    def forward(self, x):
        e0_feat = self.enc0(x)                     # [B, 64, H/2, W/2] -> [B, 64, 128, 128]
        e0 = self.maxpool(e0_feat)                 # [B, 64, H/4, W/4] -> [B, 64, 64, 64]
        e1 = self.enc1(e0)                         # [B, 256, 64, 64]
        e2 = self.enc2(e1)                         # [B, 512, 32, 32]
        e3 = self.enc3(e2)                         # [B, 1024, 16, 16]
        e4 = self.enc4(e3)                         # [B, 2048, 8, 8]
        d4 = self.dec4(torch.cat([self.up4(e4), e3], 1))       # [B, 1280, 16, 16] -> [B, 256, 16, 16]
        d3 = self.dec3(torch.cat([self.up3(d4), e2], 1))       # [B, 640, 32, 32]  -> [B, 128, 32, 32]
        d2 = self.dec2(torch.cat([self.up2(d3), e1], 1))       # [B, 320, 64, 64]  -> [B, 64, 64, 64]
        d1 = self.dec1(torch.cat([self.up1(d2), e0_feat], 1))  # [B, 128, 128, 128] -> [B, 64, 128, 128]
        f = self.final_block(self.final_up(d1))                # [B, 32, 256, 256]
        return self.out_conv(f)

class WindowAttention(nn.Module):
    def __init__(self, dim, window_size=16, num_heads=4):
        super().__init__()
        self.window_size = window_size
        self.attn = nn.MultiheadAttention(dim, num_heads, batch_first=True)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        B, C, H, W = x.shape
        ws = self.window_size
        x_windows = x.view(B, C, H // ws, ws, W // ws, ws).permute(0, 2, 4, 3, 5, 1).contiguous().view(-1, ws * ws, C)
        normed = self.norm(x_windows)
        attn_out, _ = self.attn(normed, normed, normed)
        out = (x_windows + attn_out).view(B, H // ws, W // ws, ws, ws, C).permute(0, 5, 1, 3, 2, 4).contiguous()
        return out.view(B, C, H, W)

class SEBlock(nn.Module):
    def __init__(self, ch, reduction=8):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(ch, ch // reduction), nn.ReLU(inplace=True),
            nn.Linear(ch // reduction, ch), nn.Sigmoid()
        )
    def forward(self, x):
        B, C, _, _ = x.shape
        return x * self.fc(self.pool(x).view(B, C)).view(B, C, 1, 1)

class ResBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.conv1 = nn.Conv2d(ch, ch, 3, padding=1)
        self.conv2 = nn.Conv2d(ch, ch, 3, padding=1)
        self.act = nn.ReLU(inplace=True)
    def forward(self, x):
        return x + self.conv2(self.act(self.conv1(x)))

class SRBackbone(nn.Module):
    def __init__(self, in_ch=3, feat_ch=96, n_resblocks=8):
        super().__init__()
        self.stem = nn.Conv2d(in_ch, feat_ch, 3, padding=1)
        self.body = nn.Sequential(*[ResBlock(feat_ch) for _ in range(n_resblocks)])
        self.up1 = nn.Sequential(nn.Conv2d(feat_ch, feat_ch * 4, 3, padding=1), nn.PixelShuffle(2), nn.ReLU(inplace=True))
        self.up2 = nn.Sequential(nn.Conv2d(feat_ch, feat_ch * 4, 3, padding=1), nn.PixelShuffle(2), nn.ReLU(inplace=True))

    def forward(self, x):
        feat = self.stem(x)
        feat = self.body(feat) + feat
        return self.up2(self.up1(feat))

class AdaptiveSRGenerator(nn.Module):
    def __init__(self, feat_ch=96, window_size=16):
        super().__init__()
        self.backbone = SRBackbone(feat_ch=feat_ch, n_resblocks=8)
        self.heavy_attn = WindowAttention(feat_ch, window_size=window_size)
        self.heavy_se = SEBlock(feat_ch)
        self.heavy_conv = nn.Conv2d(feat_ch, feat_ch, 3, padding=1)
        self.light_conv = nn.Sequential(nn.Conv2d(feat_ch, feat_ch, 3, padding=1), nn.ReLU(inplace=True))
        self.out_conv = nn.Conv2d(feat_ch, 3, 3, padding=1)

    def forward(self, lr, importance_map):
        feat = self.backbone(lr)
        heavy = self.heavy_conv(self.heavy_se(self.heavy_attn(feat)))
        light = self.light_conv(feat)
        blended = importance_map * heavy + (1 - importance_map) * light
        out = self.out_conv(blended)
        base = F.interpolate(lr, scale_factor=SCALE, mode='bicubic', align_corners=False)
        return torch.clamp(base + out, 0, 1)

class BaselineSRGenerator(nn.Module):
    def __init__(self, feat_ch=96, window_size=16):
        super().__init__()
        self.backbone = SRBackbone(feat_ch=feat_ch, n_resblocks=8)
        self.attn = WindowAttention(feat_ch, window_size=window_size)
        self.se = SEBlock(feat_ch)
        self.conv = nn.Conv2d(feat_ch, feat_ch, 3, padding=1)
        self.out_conv = nn.Conv2d(feat_ch, 3, 3, padding=1)

    def forward(self, lr):
        feat = self.backbone(lr)
        feat = self.conv(self.se(self.attn(feat)))
        out = self.out_conv(feat)
        base = F.interpolate(lr, scale_factor=SCALE, mode='bicubic', align_corners=False)
        return torch.clamp(base + out, 0, 1)

class Lion(optim.Optimizer):
    def __init__(self, params, lr=1e-4, betas=(0.9, 0.99), weight_decay=0.0):
        defaults = dict(lr=lr, betas=betas, weight_decay=weight_decay)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad(): loss = closure()
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None: continue
                grad = p.grad
                state = self.state[p]
                if len(state) == 0: state['exp_avg'] = torch.zeros_like(p)
                exp_avg = state['exp_avg']
                beta1, beta2 = group['betas']
                if group['weight_decay'] > 0: p.mul_(1.0 - group['lr'] * group['weight_decay'])
                update = exp_avg.mul(beta1).add(grad, alpha=1.0 - beta1).sign_()
                p.add_(update, alpha=-group['lr'])
                exp_avg.mul_(beta2).add_(grad, alpha=1.0 - beta2)
        return loss

In [ ]:
from skimage.metrics import structural_similarity as ssim_fn

def sobel_edges(img):
    gray = 0.299*img[:,0:1] + 0.587*img[:,1:2] + 0.114*img[:,2:3]
    sx = torch.tensor([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=img.dtype, device=img.device).view(1,1,3,3)
    sy = torch.tensor([[-1,-2,-1],[0,0,0],[1,2,1]], dtype=img.dtype, device=img.device).view(1,1,3,3)
    gx = F.conv2d(gray, sx, padding=1)
    gy = F.conv2d(gray, sy, padding=1)
    return torch.sqrt(gx**2 + gy**2 + 1e-6)

def pixel_loss(sr, hr, importance_map, base_weight=1.0, importance_weight=1.0):
    return (torch.abs(sr - hr) * (base_weight + importance_weight * importance_map)).mean()

def edge_loss(sr, hr, importance_map):
    diff = torch.abs(sobel_edges(sr) - sobel_edges(hr))
    return (diff * importance_map).sum() / (importance_map.sum() + 1e-6)

ce_loss = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)

def seg_guided_loss(sr, mask, seg_model):
    logits = seg_model(sr)
    return ce_loss(logits, mask)

def total_generator_loss(sr, hr, mask, importance_map, seg_model):
    l_pixel = pixel_loss(sr, hr, importance_map)
    l_edge = edge_loss(sr, hr, importance_map)
    l_seg = seg_guided_loss(sr, mask, seg_model)
    return l_pixel + 0.5 * l_edge + 0.2 * l_seg

def psnr(sr, hr):
    mse = F.mse_loss(sr, hr).item()
    return 100.0 if mse == 0 else 10 * np.log10(1.0 / mse)

def to_numpy_img(t):
    return t.permute(1, 2, 0).cpu().numpy()

def batch_ssim(sr, hr):
    scores = []
    for i in range(sr.size(0)):
        s, h = to_numpy_img(sr[i]), to_numpy_img(hr[i])
        try:
            val = ssim_fn(h, s, channel_axis=2, data_range=1.0)
        except TypeError:
            val = ssim_fn(h, s, multichannel=True, data_range=1.0)
        scores.append(val)
    return np.mean(scores)

def compute_miou(pred_logits, target, num_classes=NUM_CLASSES, ignore_index=IGNORE_INDEX):
    pred = pred_logits.argmax(1)
    ious = []
    for c in range(num_classes):
        if c == ignore_index: continue
        pred_c, target_c = (pred == c), (target == c)
        inter, union = (pred_c & target_c).sum().item(), (pred_c | target_c).sum().item()
        if union > 0: ious.append(inter / union)
    return sum(ious) / len(ious) if ious else 0.0

In [ ]:
ckpt_search = sorted(glob.glob('/kaggle/input/**/checkpoints_resnet50', recursive=True))
if ckpt_search:
    INPUT_CKPT_DIR = ckpt_search[-1]
else:
    INPUT_CKPT_DIR = '/kaggle/input/updated50/checkpoints_resnet50'

WORKING_CKPT_DIR = '/kaggle/working/checkpoints_resnet50'
os.makedirs(WORKING_CKPT_DIR, exist_ok=True)

!cp -r {INPUT_CKPT_DIR}/* {WORKING_CKPT_DIR}/
print(f"Checkpoints copied successfully from {INPUT_CKPT_DIR} -> {WORKING_CKPT_DIR}:")
print(os.listdir(WORKING_CKPT_DIR))

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

rin = RegionImportanceNet().to(device)
rin.load_state_dict(torch.load(os.path.join(WORKING_CKPT_DIR, 'rin_best.pth'), weights_only=False)['model_state_dict'])
rin.eval()
for p in rin.parameters(): p.requires_grad = False

seg_model = ResNetSegmenter(num_classes=NUM_CLASSES, backbone_name='resnet50').to(device)
seg_model.load_state_dict(torch.load(os.path.join(WORKING_CKPT_DIR, 'segmenter_best.pth'), weights_only=False)['model_state_dict'])
seg_model.eval()
for p in seg_model.parameters(): p.requires_grad = False

print("RIN and Segmenter loaded and frozen.")

In [ ]:
SR_BATCH_SIZE = 8
sr_train_loader = DataLoader(train_dataset, batch_size=SR_BATCH_SIZE, shuffle=True,
                              num_workers=4, pin_memory=True, drop_last=True)
sr_val_loader   = DataLoader(val_dataset, batch_size=SR_BATCH_SIZE, shuffle=False,
                              num_workers=4, pin_memory=True)

model = AdaptiveSRGenerator(feat_ch=96).to(device)
model_name = 'adaptive_sr_lion'
ckpt_path = os.path.join(WORKING_CKPT_DIR, f'{model_name}_latest.pth')
best_path = os.path.join(WORKING_CKPT_DIR, f'{model_name}_best.pth')

opt = Lion(model.parameters(), lr=3e-5)
sched = optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=3)

start_epoch = 1
best_psnr = 0.0
epochs = 80

if os.path.exists(ckpt_path):
    ck = torch.load(ckpt_path, weights_only=False)
    model.load_state_dict(ck['model_state_dict'])
    opt.load_state_dict(ck['optimizer_state_dict'])
    start_epoch = ck['epoch'] + 1
    best_psnr = ck.get('best_psnr', 0.0)
    print(f"[{model_name}] Resuming seamlessly from epoch {start_epoch} (Previous best PSNR: {best_psnr:.2f}dB)")

if start_epoch > epochs:
    print(f"[{model_name}] Training already completed (Epoch {ck['epoch']}/{epochs})! Skipping training loop and proceeding straight to evaluation...")
else:
    for epoch in range(start_epoch, epochs + 1):
        model.train()
        t0 = time.time()
        running_loss = 0.0
        for batch in sr_train_loader:
            lr_img = batch['lr'].to(device, non_blocking=True)
            hr_img = batch['hr'].to(device, non_blocking=True)
            mask   = batch['mask'].to(device, non_blocking=True)

            opt.zero_grad()
            with torch.no_grad():
                importance = rin(lr_img)
            sr = model(lr_img, importance)
            loss = total_generator_loss(sr, hr_img, mask, importance, seg_model)
            loss.backward()
            opt.step()
            running_loss += loss.item() * lr_img.size(0)

        train_loss = running_loss / len(train_dataset)

        model.eval()
        val_psnr_sum, n = 0.0, 0
        with torch.no_grad():
            for batch in sr_val_loader:
                lr_img = batch['lr'].to(device, non_blocking=True)
                hr_img = batch['hr'].to(device, non_blocking=True)
                importance = rin(lr_img)
                sr = model(lr_img, importance)
                val_psnr_sum += psnr(sr, hr_img) * lr_img.size(0)
                n += lr_img.size(0)
        val_psnr = val_psnr_sum / n
        sched.step(-val_psnr)

        elapsed = time.time() - t0
        print(f"[{model_name}] Epoch {epoch:02d}/{epochs} | train_loss {train_loss:.4f} | val_PSNR {val_psnr:.2f}dB | {elapsed:.1f}s")

        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': opt.state_dict(), 'best_psnr': max(best_psnr, val_psnr)}, ckpt_path)
        if val_psnr > best_psnr:
            best_psnr = val_psnr
            torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(), 'val_psnr': val_psnr}, best_path)
            print(f"  -> saved new best {model_name} (PSNR={val_psnr:.2f}dB)")

    print(f"\nStage 4b complete! Final Lion Best PSNR: {best_psnr:.2f} dB")

In [ ]:
print("\n" + "="*70)
print("RUNNING FINAL ABLATION & DOWNSTREAM EVALUATION")
print("="*70)

baseline_gen = BaselineSRGenerator(feat_ch=96).to(device)
baseline_gen.load_state_dict(torch.load(os.path.join(WORKING_CKPT_DIR, 'baseline_sr_best.pth'), weights_only=False)['model_state_dict'])
baseline_gen.eval()

adaptive_adamw_gen = AdaptiveSRGenerator(feat_ch=96).to(device)
adaptive_adamw_gen.load_state_dict(torch.load(os.path.join(WORKING_CKPT_DIR, 'adaptive_sr_adamw_best.pth'), weights_only=False)['model_state_dict'])
adaptive_adamw_gen.eval()

adaptive_lion_gen = AdaptiveSRGenerator(feat_ch=96).to(device)
lion_ckpt_path = os.path.join(WORKING_CKPT_DIR, 'adaptive_sr_lion_best.pth')
if not os.path.exists(lion_ckpt_path):
    lion_ckpt_path = os.path.join(WORKING_CKPT_DIR, 'adaptive_sr_lion_latest.pth')
adaptive_lion_gen.load_state_dict(torch.load(lion_ckpt_path, weights_only=False)['model_state_dict'])
adaptive_lion_gen.eval()

test_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=4)

results = {
    'Bicubic':       {'psnr': [], 'ssim': [], 'miou': []},
    'Baseline SR':   {'psnr': [], 'ssim': [], 'miou': []},
    'Adaptive AdamW':{'psnr': [], 'ssim': [], 'miou': []},
    'Adaptive Lion': {'psnr': [], 'ssim': [], 'miou': []},
}

with torch.no_grad():
    for batch in test_loader:
        lr_img = batch['lr'].to(device)
        hr_img = batch['hr'].to(device)
        mask   = batch['mask'].to(device)

        bicubic_sr    = F.interpolate(lr_img, scale_factor=SCALE, mode='bicubic', align_corners=False).clamp(0,1)
        baseline_sr   = baseline_gen(lr_img)
        imp           = rin(lr_img)
        adaptive_adam = adaptive_adamw_gen(lr_img, imp)
        adaptive_lion = adaptive_lion_gen(lr_img, imp)

        for name, sr in [('Bicubic', bicubic_sr), ('Baseline SR', baseline_sr), 
                         ('Adaptive AdamW', adaptive_adam), ('Adaptive Lion', adaptive_lion)]:
            results[name]['psnr'].append(psnr(sr, hr_img))
            results[name]['ssim'].append(batch_ssim(sr, hr_img))
            results[name]['miou'].append(compute_miou(seg_model(sr), mask))

print("\n" + "="*70)
print(f"{'Method':<20} {'PSNR (dB)':<12} {'SSIM':<10} {'Seg mIoU':<10}")
print("="*70)
for name in ['Bicubic', 'Baseline SR', 'Adaptive AdamW', 'Adaptive Lion']:
    r = results[name]
    print(f"{name:<20} {np.mean(r['psnr']):<12.2f} {np.mean(r['ssim']):<10.4f} {np.mean(r['miou']):<10.4f}")
print("="*70)

# Generate Comparison Grid
sample_batch = next(iter(test_loader))
lr_img = sample_batch['lr'][:4].to(device)
hr_img = sample_batch['hr'][:4].to(device)

with torch.no_grad():
    imp = rin(lr_img)
    bicubic_sr    = F.interpolate(lr_img, scale_factor=SCALE, mode='bicubic', align_corners=False).clamp(0,1)
    baseline_sr   = baseline_gen(lr_img)
    adaptive_adam = adaptive_adamw_gen(lr_img, imp)
    adaptive_lion = adaptive_lion_gen(lr_img, imp)

fig, axes = plt.subplots(4, 6, figsize=(24, 16))
titles = ['LR Input', 'Bicubic', 'Baseline SR', 'Adaptive (AdamW)', 'Adaptive (Lion)', 'Ground Truth HR']

for row in range(4):
    imgs = [lr_img[row], bicubic_sr[row], baseline_sr[row], adaptive_adam[row], adaptive_lion[row], hr_img[row]]
    for col, (img, title) in enumerate(zip(imgs, titles)):
        axes[row, col].imshow(to_numpy_img(img))
        if row == 0: axes[row, col].set_title(title, fontsize=12)
        axes[row, col].axis('off')

plt.tight_layout()
grid_path = '/kaggle/working/comparison_grid_final.png'
plt.savefig(grid_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Final comparison grid saved to {grid_path}!")